# Latent world model for predictive collision avoidance (Go2, V-JEPA 2.1)

Training and offline evaluation of an **action-conditioned latent world model** for a Unitree Go2 quadruped on simulated construction sites (Isaac Lab). The closed-loop planner that uses the saved model is in `wm/planner.py`.

**Pipeline**
1. **Frozen encoder: V-JEPA 2.1 ViT-L/16, 384 px.** Each pair of frames (t−1, t) gives 24×24 tokens of 1024 dims. They are pooled to a 4×4 grid (stored) and then a 2×2 grid (used), so the model still knows *where* things are.
2. **Action-conditioned predictor (the world model).** From the last 8 latent frames (1.6 s) and the commands (vx, vy, ωz), a block-causal transformer predicts the next latent frame. Actions enter every layer through AdaLN (LeWM / Orbit-Planner design). Training uses teacher forcing plus a rollout loss, on the main trajectories **and the counterfactual branches**.
3. **Danger head.** Reads "danger within 3 s" from a latent frame, real or imagined.
4. **Planning (latent MPC).** Imagine 3 s for candidate command sequences, score each with the danger head and the progress to the goal, and execute the first command of the best one.

**Evaluation**
- Danger detection on real latents: all frames, anticipation cases, per behaviour.
- Prediction error vs horizon, with controls (shuffled actions, no actions) and action sensitivity by distance.
- Counterfactual prediction: does the future imagined for command *k* match the real branch of *k*?
- Offline planning on branch points: the planner picks a command, and we read its real outcome from the branch.

**Data**: simulated dataset from the project partners (private, not included). 500 episodes and 155k samples, plus counterfactual branches (4,431 states × 7 commands × 3 s).

**Run**: Colab with an A100, top to bottom. Features are cached on Drive.

## 0. Config

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules

CAMPAIGN      = "excavator_proto_v4_9"
HF_REPO       = None          # dataset repo id (private, ask the project owners)
HF_TOKEN      = None          # use a Colab secret named HF_TOKEN
DATA_ROOT     = "/content/data_hf" if IN_COLAB else "./data_hf"
USE_DRIVE     = IN_COLAB      # cache features / results on Google Drive
WORK          = "/content/drive/MyDrive/go2_wm" if USE_DRIVE else "./go2_wm"

ENCODER       = "vjepa2_1_vitl"   # "stub" = random tiny encoder for a CPU smoke test
IMG           = 384               # V-JEPA 2.1 resolution
FRAME_STRIDE  = 2                 # keep 1 frame out of 2 -> 5 Hz
BATCH         = 32                # encoder batch (clips); lower if out of memory
GRID_STORE    = 4                 # tokens stored per frame: 4x4 spatial grid
GRID          = 2                 # tokens used by the models: 2x2 (pooled from 4x4). Try 4 later.
ENCODE_BRANCHES = True            # also encode the counterfactual branch videos
MAX_EPISODES  = None              # e.g. 20 for a quick test

CTX           = 8                 # context frames (1.6 s at 5 Hz)
H             = 15                # predicted steps (3 s at 5 Hz)
FALL_CUT_S    = 2.0               # drop the last seconds before a fall
TRAIN_BS      = None              # predictor batch; None = 512 // tokens per frame (128 for 2x2, 32 for 4x4)
MARGIN_M      = 1.0               # danger margin used for branch outcomes (same as danger_H)
SEED          = 0

# command limits of the Go2 walking policy (from Insu's CONFIG.md); the planner never leaves them
A_LOW  = [-0.3, -0.25, -0.6]
A_HIGH = [ 0.9,  0.25,  0.6]
PRIMITIVES = {                    # the 7 branch commands of v4.9
    "forward": [0.7, 0.0, 0.0], "turn_left": [0.3, 0.0, 0.5], "turn_right": [0.3, 0.0, -0.5],
    "wait": [0.0, 0.0, 0.0], "backward": [-0.3, 0.0, 0.0],
    "strafe_left": [0.0, 0.25, 0.0], "strafe_right": [0.0, -0.25, 0.0]}

## 1. Setup

In [ ]:
import subprocess
def sh(cmd): print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout[-2000:])
if IN_COLAB:
    sh("nvidia-smi -L")
    sh("pip -q install timm einops av huggingface_hub scikit-learn")
    if not os.path.exists("/content/vjepa2"):
        sh("git clone -q --depth 1 https://github.com/facebookresearch/vjepa2 /content/vjepa2")
    sys.path.insert(0, "/content/vjepa2")

In [ ]:
import json, glob, tarfile, math, random, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
FEAT_DIR    = f"{WORK}/features/{CAMPAIGN}/{ENCODER}_g{GRID_STORE}_s{FRAME_STRIDE}"
RESULTS_DIR = f"{WORK}/results/{CAMPAIGN}"
os.makedirs(FEAT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
DT  = 0.1 * FRAME_STRIDE
print("device:", DEV, "| features:", FEAT_DIR)

## 2. Download and unpack the dataset

In [ ]:
CAMP_DIR = os.path.join(DATA_ROOT, CAMPAIGN)
if IN_COLAB and not os.path.exists(os.path.join(CAMP_DIR, "episodes")):
    assert HF_REPO, "set HF_REPO (the dataset is private)"
    from huggingface_hub import login, snapshot_download
    tok = HF_TOKEN
    if tok is None:
        try:
            from google.colab import userdata; tok = userdata.get("HF_TOKEN")
        except Exception:
            tok = None
    login(token=tok) if tok else login()
    snapshot_download(repo_id=HF_REPO, repo_type="dataset", allow_patterns=[f"{CAMPAIGN}/*"],
                      local_dir=DATA_ROOT, token=tok)
    for f in sorted(glob.glob(os.path.join(CAMP_DIR, "data", "*.tar"))):
        with tarfile.open(f) as t: t.extractall(CAMP_DIR)
EP_DIRS = sorted(glob.glob(os.path.join(CAMP_DIR, "episodes", "ep_*")))
if MAX_EPISODES: EP_DIRS = EP_DIRS[:MAX_EPISODES]
print(len(EP_DIRS), "episodes in", CAMP_DIR)

## 3. Data check: one episode and its branches

In [ ]:
def read_jsonl(p):
    with open(p) as f: return [json.loads(l) for l in f if l.strip()]

ep_b = next((e for e in EP_DIRS if os.path.exists(os.path.join(e, "branches", "branches.json"))), EP_DIRS[0])
print("campaign folder:", sorted(os.listdir(CAMP_DIR)))
print("episode with branches:", ep_b, "|", sorted(os.listdir(ep_b)))
meta0 = json.load(open(os.path.join(ep_b, "metadata.json")))
print("metadata keys:", list(meta0)); print("branching:", {k: v for k, v in meta0.get("branching", {}).items() if k not in ("near", "candidates")})
print("trajectory keys:", list(read_jsonl(os.path.join(ep_b, "trajectory.jsonl"))[0]))
print("label keys:", list(read_jsonl(os.path.join(ep_b, "labels_h3_m1.jsonl"))[0]))

## 4. Per-episode data (labels, commands, pose, goal)

In [ ]:
def yaw_from_quat(q):   # xyzw
    x, y, z, w = q
    return math.atan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))

def fnum(r, k, default=np.nan):
    v = r.get(k, None)
    return default if v is None else float(v)

def episode_arrays(ep):
    meta = json.load(open(os.path.join(ep, "metadata.json")))
    traj = read_jsonl(os.path.join(ep, "trajectory.jsonl"))
    lab  = read_jsonl(os.path.join(ep, "labels_h3_m1.jsonl"))
    n = min(len(traj), len(lab))
    traj, lab = traj[:n], lab[:n]
    y      = np.array([fnum(r, "danger_H") for r in lab], np.float32)
    valid  = np.array([bool(r.get("valid", True)) for r in lab]) & ~np.isnan(y)
    antic  = np.array([fnum(r, "anticipation_H", 0.0) for r in lab], np.float32)
    clear  = np.array([fnum(r, "clearance_now_m") for r in lab], np.float32)
    act    = np.array([r["command"][:3] for r in traj], np.float32)
    pos    = np.array([r["robot"]["pos"][:2] for r in traj], np.float32)
    yaw    = np.array([yaw_from_quat(r["robot"]["quat_xyzw"]) for r in traj], np.float32)
    sim_step = np.array([r.get("sim_step", i) for i, r in enumerate(traj)])
    term = str(meta.get("termination", ""))
    usable = np.ones(n, bool)                       # frames allowed in predictor windows
    if term == "fall":
        cut = int(FALL_CUT_S / 0.1); usable[max(0, n - cut):] = False; valid[max(0, n - cut):] = False
    goal = meta.get("episode", {}).get("goal_xy")
    return dict(name=os.path.basename(ep), y=y, valid=valid, antic=antic, clear=clear, act=act, pos=pos,
                yaw=yaw, sim_step=sim_step, usable=usable, goal=None if goal is None else np.array(goal, np.float32),
                behav=str(meta.get("behaviour", "?")), scen=str(meta.get("scenario", "?")), term=term)

t0 = time.time()
EPS = [episode_arrays(ep) for ep in EP_DIRS]
print(f"loaded {len(EPS)} episodes in {time.time() - t0:.0f}s")
allv = np.concatenate([e["y"][e["valid"]] for e in EPS])
print(f"valid frames {len(allv)}, danger {allv.mean():.3f}, falls cut in {sum(e['term'] == 'fall' for e in EPS)} episodes")

## 5. Counterfactual branches
Each branch point is a saved simulator state where the robot was 0.5 to 5 m from a hazard. From it, each of the 7 commands is replayed for 3 s (`branches/branches.json` plus one video per command, `b{k:03d}_{cmd}.mp4`). A branch is **unsafe** if the robot comes within 1 m of a machine, touches it, or falls.

In [ ]:
# ===== 5. Counterfactual branches (v4.9: branches/branches.json + b{k:03d}_{cmd}.mp4) =====
from collections import Counter

def list_branches(ep):
    bj = os.path.join(ep, "branches", "branches.json")
    if not os.path.exists(bj): return []
    pts = []
    for b in json.load(open(bj)):
        units = []
        for name, c in b["candidates"].items():
            S = c.get("samples") or []
            cl = np.array([np.nan if s.get("clearance_m") is None else s["clearance_m"] for s in S], np.float32)
            bad = np.array([bool(s.get("machine_foot_contact") or s.get("nonfoot_contact") or s.get("fallen")) for s in S], bool)
            d = (np.nan_to_num(cl, nan=1e9) < MARGIN_M) | bad            # danger flag per 10 Hz sample of the branch
            rgb = os.path.join(ep, "branches", f"b{int(b['k']):03d}_{name}.mp4")
            units.append(dict(name=name, cmd=np.array(c["command"][:3], np.float32),
                              rgb=rgb if os.path.exists(rgb) else None, outcome=str(c.get("outcome")),
                              cmin=float(np.nanmin(cl)) if np.isfinite(cl).any() else np.nan,
                              unsafe=bool(d.any()), d=d))
        pts.append(dict(t0=int(b["main_index"]), k=int(b["k"]), trigger=b.get("trigger"), units=units))
    return pts

BRANCHES, OUTC, t_ = {}, Counter(), time.time()
for i, ep in enumerate(EP_DIRS):
    e = EPS[i]; good = []
    for p in list_branches(ep):
        if not ((CTX - 1) * FRAME_STRIDE <= p["t0"] < len(e["y"])) or not e["usable"][p["t0"]]: continue
        OUTC.update(u["outcome"] for u in p["units"]); good.append(p)
    if good: BRANCHES[i] = good
allp = [p for v in BRANCHES.values() for p in v]
print(f"branch points: {len(allp)} in {len(BRANCHES)} episodes ({time.time() - t_:.0f}s) | outcomes: {dict(OUTC)}")
print("decisive points (some commands safe, some not):", sum(0 < sum(u["unsafe"] for u in p["units"]) < len(p["units"]) for p in allp),
      "| all unsafe:", sum(all(u["unsafe"] for u in p["units"]) for p in allp))
print("unsafe rate per command:", {n: round(float(np.mean([u["unsafe"] for p in allp for u in p["units"] if u["name"] == n])), 2) for n in PRIMITIVES})

## 6. Encoder: V-JEPA 2.1 ViT-L/16, 384 px (frozen)

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def prep(frames_uint8, size):
    x = torch.from_numpy(frames_uint8).permute(0, 3, 1, 2).float() / 255.0
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False, antialias=True)
    return (x - MEAN) / STD

def grid_pool(tokens, g):
    # (B, N, D) tokens on a square grid -> (B, g*g, D)
    B, N, D = tokens.shape; h = int(round(math.sqrt(N)))
    x = tokens[:, -h * h:].reshape(B, h, h, D).permute(0, 3, 1, 2).float()
    return F.adaptive_avg_pool2d(x, g).flatten(2).transpose(1, 2)

class VJepa21:
    name = "vjepa2_1_vitl"
    def __init__(self):
        import src.hub.backbones as bb
        bb.VJEPA_BASE_URL = "https://dl.fbaipublicfiles.com/vjepa2"   # the repo points to localhost by default
        enc, pred = bb.vjepa2_1_vit_large_384(pretrained=True)
        del pred
        self.m = enc.to(DEV).eval()
    @torch.no_grad()
    def __call__(self, frames, prev):
        # clip = (frame t-1, frame t): one 2-frame tubelet -> 24x24 tokens of 1024 dims
        clip = torch.stack([prep(prev, IMG), prep(frames, IMG)], 2).to(DEV)              # (B, C, T=2, H, W)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEV == "cuda")):
            tok = self.m(clip)
        if isinstance(tok, (list, tuple)): tok = tok[-1]
        return grid_pool(tok, GRID_STORE)                                                  # (B, G*G, 1024)

class StubEncoder:      # CPU smoke test only
    name = "stub"
    def __init__(self):
        torch.manual_seed(0); self.w = torch.randn(3 * 16 * 16, 64) / 30
    @torch.no_grad()
    def __call__(self, frames, prev):
        x = prep(frames, 64) - 0.5 * prep(prev, 64)
        p = F.unfold(x, 16, stride=16).transpose(1, 2) @ self.w        # (B, 16, 64)
        return grid_pool(p, GRID_STORE)

ENC = VJepa21() if ENCODER.startswith("vjepa") else StubEncoder()
z = ENC(np.zeros((2, 320, 320, 3), np.uint8), np.zeros((2, 320, 320, 3), np.uint8))
print("token grid per frame:", tuple(z.shape))

## 7. Feature extraction (main trajectories), cached on Drive

In [ ]:
def load_frames(mp4):
    import av
    with av.open(mp4) as c:
        return np.stack([f.to_ndarray(format="rgb24") for f in c.decode(video=0)])

def encode_video(frames, idx):
    prev = np.maximum(idx - 1, 0); out = []
    for s in range(0, len(idx), BATCH):
        out.append(ENC(frames[idx[s:s + BATCH]], frames[prev[s:s + BATCH]]).half().cpu().numpy())
    return np.concatenate(out)

t0 = time.time()
for i, ep in enumerate(EP_DIRS):
    dst = os.path.join(FEAT_DIR, EPS[i]["name"] + ".npz")
    if os.path.exists(dst): continue
    fr = load_frames(os.path.join(ep, "rgb.mp4"))
    n = min(len(fr), len(EPS[i]["y"]))
    idx = np.arange(0, n, FRAME_STRIDE)
    np.savez(dst, tok=encode_video(fr, idx), idx=idx)
    if i % 20 == 0: print(f"main {i + 1}/{len(EP_DIRS)}  {time.time() - t0:.0f}s")
print(f"main trajectories done in {time.time() - t0:.0f}s")

## 8. Load features, normalise, split by episode

In [ ]:
def pool_to(tok, g):
    if g == GRID_STORE: return tok
    B, N, D = tok.shape; s = GRID_STORE
    x = torch.from_numpy(tok.astype(np.float32)).reshape(B, s, s, D).permute(0, 3, 1, 2)
    return F.adaptive_avg_pool2d(x, g).flatten(2).transpose(1, 2).numpy().astype(np.float16)

for i, e in enumerate(EPS):
    f = np.load(os.path.join(FEAT_DIR, e["name"] + ".npz"))
    idx = f["idx"]; e["z"] = pool_to(f["tok"], GRID)                      # (T5, P, D) fp16
    e["idx"] = idx
    for k in ["y", "valid", "antic", "clear", "pos", "yaw", "usable"]:
        e[k + "5"] = e[k][idx]
    a = e["act"]                                                         # command held over each 5 Hz step
    e["act5"] = np.stack([a[t:t + FRAME_STRIDE].mean(0) for t in idx]).astype(np.float32)
P, D = EPS[0]["z"].shape[1:]
print("latent frame:", P, "tokens x", D, "dims")

groups = {}
for i, e in enumerate(EPS): groups.setdefault((e["behav"], e["scen"]), []).append(i)
rng = np.random.default_rng(SEED); SPLIT = {}
for g, ids_ in groups.items():
    ids_ = list(rng.permutation(ids_)); n = len(ids_)
    n_te = max(1, round(0.15 * n)) if n >= 3 else 0; n_va = max(1, round(0.15 * n)) if n >= 3 else 0
    for j, i in enumerate(ids_): SPLIT[i] = "test" if j < n_te else ("val" if j < n_te + n_va else "train")
IDS = {s: [i for i, v in SPLIT.items() if v == s] for s in ["train", "val", "test"]}
print({s: len(v) for s, v in IDS.items()})

zs = np.concatenate([EPS[i]["z"][::3].reshape(-1, D).astype(np.float32) for i in IDS["train"]])
Z_MU, Z_SD = zs.mean(0), zs.std(0) + 1e-6; del zs
acts = np.concatenate([EPS[i]["act5"] for i in IDS["train"]])
A_MU, A_SD = acts.mean(0), acts.std(0) + 1e-6
for e in EPS: e["zn"] = ((e["z"].astype(np.float32) - Z_MU) / Z_SD).astype(np.float16)
na = lambda a: (np.asarray(a, np.float32) - A_MU) / A_SD
print("action mean", A_MU.round(2), "std", A_SD.round(2))

## 8b. Encode the branch videos (subset per split)

In [ ]:
# branch points encoded per episode: all would be ~470k clips, this subset ~250k
BR_PTS = {"train": 6, "val": 3, "test": 3}     # branch points encoded per episode; None = all (much slower)

def encode_many(videos):
    cur, prv, lens = [], [], []
    for fr in videos:
        idx = np.arange(0, len(fr), FRAME_STRIDE)
        cur.append(fr[idx]); prv.append(fr[np.maximum(idx - 1, 0)]); lens.append(len(idx))
    cur, prv = np.concatenate(cur), np.concatenate(prv); out = []
    for s in range(0, len(cur), BATCH):
        out.append(ENC(cur[s:s + BATCH], prv[s:s + BATCH]).half().cpu().numpy())
    return np.split(np.concatenate(out), np.cumsum(lens)[:-1])

todo = [i for i in BRANCHES if not os.path.exists(os.path.join(FEAT_DIR, EPS[i]["name"] + "_branches.npz"))]
print(len(todo), "episodes to encode"); t_ = time.time()
for k, i in enumerate(todo):
    pts = BRANCHES[i]; cap = BR_PTS[SPLIT[i]]
    js = range(len(pts)) if cap is None else sorted(np.random.default_rng(i).permutation(len(pts))[:cap])
    keys, vids = [], []
    for j in js:
        for u_i, u in enumerate(pts[j]["units"]):
            if u["rgb"]: keys.append(f"p{j}_u{u_i}"); vids.append(load_frames(u["rgb"]))
    if vids: np.savez(os.path.join(FEAT_DIR, EPS[i]["name"] + "_branches.npz"), **dict(zip(keys, encode_many(vids))))
    if (k + 1) % 20 == 0: print(f"{k + 1}/{len(todo)} episodes  {time.time() - t_:.0f}s")
print(f"branch encoding done in {time.time() - t_:.0f}s")

## 9. Level 1: risk probe on real latents

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

class TokenProbe(nn.Module):
    # (B, P, D) -> logit of "danger within 3 s"; mean+max pooling over the spatial tokens
    def __init__(s, D, h=256, p=0.2):
        super().__init__()
        s.tok = nn.Sequential(nn.LayerNorm(D), nn.Linear(D, h), nn.GELU())
        s.head = nn.Sequential(nn.Dropout(p), nn.Linear(2 * h, h), nn.GELU(), nn.Linear(h, 1))
    def forward(s, z):
        h = s.tok(z); return s.head(torch.cat([h.mean(1), h.max(1).values], -1)).squeeze(-1)

def frames_xy(ids_):
    X, Y, A, B = [], [], [], []
    for i in ids_:
        e = EPS[i]; m = e["valid5"]
        X.append(e["zn"][m]); Y.append(e["y5"][m]); A.append(e["antic5"][m]); B += [e["behav"]] * int(m.sum())
    return np.concatenate(X), np.concatenate(Y), np.concatenate(A), np.array(B)

def train_probe(Xtr, Ytr, Xva, Yva, epochs=12, lr=1e-3, bs=512):
    m = TokenProbe(D).to(DEV); opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-2)
    pw = torch.tensor((1 - Ytr.mean()) / max(Ytr.mean(), 1e-6), device=DEV)
    best, state = -1, None
    for ep in range(epochs):
        m.train(); perm = np.random.permutation(len(Xtr))
        for s in range(0, len(perm), bs):
            b = perm[s:s + bs]
            x = torch.from_numpy(Xtr[b].astype(np.float32)).to(DEV); y = torch.from_numpy(Ytr[b]).to(DEV)
            loss = F.binary_cross_entropy_with_logits(m(x), y, pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
        auc = roc_auc_score(Yva, probe_predict(m, Xva)) if len(set(Yva)) > 1 else -1
        if auc > best: best, state = auc, {k: v.clone() for k, v in m.state_dict().items()}
    m.load_state_dict(state); m.eval(); return m

@torch.no_grad()
def probe_predict(m, X, bs=2048):
    m.eval(); out = []
    for s in range(0, len(X), bs):
        x = X[s:s + bs]; x = torch.as_tensor(np.asarray(x, np.float32) if isinstance(x, np.ndarray) else x.float())
        out.append(torch.sigmoid(m(x.to(DEV))).cpu().numpy())
    return np.concatenate(out)

def report(name, Y, P_, extra=None):
    r = dict(model=name, auroc=roc_auc_score(Y, P_), ap=average_precision_score(Y, P_), pos=float(Y.mean()), n=len(Y))
    if extra: r.update(extra)
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()}); return r

RESULTS = []
Xtr, Ytr, _, _ = frames_xy(IDS["train"]); Xva, Yva, _, _ = frames_xy(IDS["val"])
Xte, Yte, Ate_, Bte = frames_xy(IDS["test"])
PROBE_REAL = train_probe(Xtr, Ytr, Xva, Yva)
pte = probe_predict(PROBE_REAL, Xte)
per_b = {f"auroc_{b}": roc_auc_score(Yte[Bte == b], pte[Bte == b]) for b in sorted(set(Bte)) if len(set(Yte[Bte == b])) > 1}
RESULTS.append(report("L1 probe | all test frames", Yte, pte, per_b))
# anticipation: danger frames that are anticipation_H vs safe frames that are far from machines
far = np.concatenate([EPS[i]["clear5"][EPS[i]["valid5"]] for i in IDS["test"]]) > 2.0
msk = far & ((Yte == 0) | (Ate_ == 1))
RESULTS.append(report("L1 probe | anticipation (robot > 2 m now)", Yte[msk], pte[msk]))
del Xtr

## 10. The world model: action-conditioned predictor (AdaLN, Orbit-Planner / LeWM style)

In [ ]:
# AdaLN-zero block adapted from Orbit-Planner (Apache-2.0, github.com/ZhijianLi2003/Orbit_Planner),
# itself from LeWM: the action embedding modulates every layer; zero init lets the action effect grow during training.
def _mod(x, shift, scale): return x * (1 + scale) + shift

class AdaLNBlock(nn.Module):
    def __init__(s, d, heads, cond_dim, mlp_ratio=4.0, p=0.1):
        super().__init__()
        s.n1 = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6); s.n2 = nn.LayerNorm(d, elementwise_affine=False, eps=1e-6)
        s.attn = nn.MultiheadAttention(d, heads, dropout=p, batch_first=True)
        s.mlp = nn.Sequential(nn.Linear(d, int(d * mlp_ratio)), nn.GELU(), nn.Dropout(p), nn.Linear(int(d * mlp_ratio), d), nn.Dropout(p))
        s.ada = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, 6 * d)); nn.init.zeros_(s.ada[-1].weight); nn.init.zeros_(s.ada[-1].bias)
    def forward(s, x, cond, mask):
        a, b, g, c, d_, h = s.ada(cond).chunk(6, -1)
        y = _mod(s.n1(x), a, b); x = x + g * s.attn(y, y, y, attn_mask=mask, need_weights=False)[0]
        return x + h * s.mlp(_mod(s.n2(x), c, d_))

class Predictor(nn.Module):
    # z (B, T, P, D) normalised latents + a (B, T, 3) commands -> prediction of z_{t+1} at every position t
    def __init__(s, D, P, a_dim=3, d=384, depth=6, heads=8, max_T=64):
        super().__init__()
        s.inp, s.out = nn.Linear(D, d), nn.Linear(d, D)
        s.pos_t = nn.Parameter(torch.zeros(1, max_T, 1, d)); s.pos_p = nn.Parameter(torch.zeros(1, 1, P, d))
        nn.init.trunc_normal_(s.pos_t, std=0.02); nn.init.trunc_normal_(s.pos_p, std=0.02)
        s.act = nn.Sequential(nn.Linear(a_dim, 128), nn.GELU(), nn.Linear(128, d))
        s.blocks = nn.ModuleList([AdaLNBlock(d, heads, d) for _ in range(depth)])
        s.norm = nn.LayerNorm(d)
    def forward(s, z, a):
        B, T, P_, _ = z.shape
        x = (s.inp(z) + s.pos_t[:, :T] + s.pos_p).reshape(B, T * P_, -1)
        cond = s.act(a)[:, :, None].expand(B, T, P_, -1).reshape(B, T * P_, -1)
        t_idx = torch.arange(T, device=z.device).repeat_interleave(P_)
        mask = torch.zeros(T * P_, T * P_, device=z.device).masked_fill(t_idx[None, :] > t_idx[:, None], float("-inf"))
        for blk in s.blocks: x = blk(x, cond, mask)
        return z + s.out(s.norm(x)).reshape(B, T, P_, -1)        # residual on z_t

def rollout(m, z_ctx, a_seq, grad=False):
    # z_ctx (B, C, P, D) real latents; a_seq (B, C+K, 3) normalised commands -> (B, K, P, D) imagined latents
    with torch.set_grad_enabled(grad):
        z = z_ctx; C = z.shape[1]; K = a_seq.shape[1] - C
        for k in range(K):
            T = z.shape[1]; s = max(0, T - CTX)
            nxt = m(z[:, s:], a_seq[:, s:T])[:, -1:]
            z = torch.cat([z, nxt], 1)
        return z[:, C:]

def window_starts(ids_, L, step):
    out = []
    for i in ids_:
        e = EPS[i]; u = e["usable5"]; T = len(e["zn"])
        for t in range(0, T - L + 1, step):
            if u[t:t + L].all(): out.append((i, t))
    return out

def get_batch(starts, L, shuffle_actions=False, zero_actions=False):
    Z = np.stack([EPS[i]["zn"][t:t + L] for i, t in starts]).astype(np.float32)
    A = np.stack([na(EPS[i]["act5"][t:t + L]) for i, t in starts]).astype(np.float32)
    if shuffle_actions: A = A[np.random.permutation(len(A))]
    if zero_actions: A = np.zeros_like(A)
    return torch.from_numpy(Z).to(DEV), torch.from_numpy(A).to(DEV)

In [ ]:
L_TRAIN = CTX + 4            # training windows: 8 context frames + 4 steps to unroll
TRAIN_STARTS = window_starts(IDS["train"], L_TRAIN, step=2)
VAL_STARTS   = window_starts(IDS["val"], CTX + H, step=5)
print(len(TRAIN_STARTS), "training windows,", len(VAL_STARTS), "validation windows")

@torch.no_grad()
def val_error(m, mode="real", n=1024):
    m.eval(); st = VAL_STARTS[:n]; errs = []
    for s in range(0, len(st), 128):
        Z, A = get_batch(st[s:s + 128], CTX + H, zero_actions=(mode == "none"))
        errs.append(F.mse_loss(rollout(m, Z[:, :CTX], A), Z[:, CTX:]).item())
    return float(np.mean(errs))

def train_predictor(mode="real", epochs=15, patience=3, lr=3e-4, bs=None, rollout_w=1.0):
    # mode: "real" executed commands | "shuffled" commands of another window (control) | "none" zeros (control)
    bs = bs or TRAIN_BS or max(16, 512 // P)
    torch.manual_seed(SEED); m = Predictor(D, P).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.05)
    best, state, bad = 1e9, None, 0
    for ep in range(epochs):
        m.train(); perm = np.random.permutation(len(TRAIN_STARTS)); t0 = time.time()
        for s in range(0, len(perm), bs):
            Z, A = get_batch([TRAIN_STARTS[j] for j in perm[s:s + bs]], L_TRAIN,
                             shuffle_actions=(mode == "shuffled"), zero_actions=(mode == "none"))
            loss = F.mse_loss(m(Z, A)[:, :-1], Z[:, 1:])                              # teacher forcing
            ro = rollout(m, Z[:, :CTX], A, grad=True)                                   # 4-step rollout loss
            loss = loss + rollout_w * F.mse_loss(ro, Z[:, CTX:])
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        err = val_error(m, mode); flag = ""
        if err < best: best, state, bad, flag = err, {k: v.clone() for k, v in m.state_dict().items()}, 0, " *"
        else: bad += 1
        print(f"[{mode}] epoch {ep:02d}  val 3 s rollout error {err:.4f}{flag}  ({time.time() - t0:.0f}s)")
        if bad >= patience: break
    m.load_state_dict(state); m.eval(); return m

WM = {"real": train_predictor("real")}
WM["shuffled"] = train_predictor("shuffled")
WM["none"] = train_predictor("none")

## 10b. World model trained on main trajectories + counterfactual branches
Without branches, long waits and turns held for 3 s are rare, so their imagined futures are out of distribution, and the danger head read them as dangerous ("waiting is risky", even far from machines). Mixing branch sequences into every batch fixes this. The `real` and `shuffled` models are retrained; `none` stays from section 10.

In [ ]:
# ===== 10b. Retrain the world model on main trajectories + counterfactual branches =====
BR_FRAC = 0.5      # share of each batch taken from branch sequences
FUT = 14           # future frames per branch at 5 Hz (3 s branch = 15 frames incl. the start frame)

def build_branch_seqs(ids_):
    # each sequence = 8 real context frames before the branch point + 14 frames of one branch, with its command
    Zs, As, Ys, meta = [], [], [], []
    for i in ids_:
        p_ = os.path.join(FEAT_DIR, EPS[i]["name"] + "_branches.npz")
        if i not in BRANCHES or not os.path.exists(p_): continue
        f = np.load(p_); keys = set(f.files); e = EPS[i]
        for j, p in enumerate(BRANCHES[i]):
            t5 = int(np.searchsorted(e["idx"], p["t0"], "right") - 1)
            if t5 < CTX - 1 or not e["usable5"][t5 - CTX + 1:t5 + 1].all(): continue
            ctx, ah = e["zn"][t5 - CTX + 1:t5 + 1], e["act5"][t5 - CTX + 1:t5]
            for u_i, u in enumerate(p["units"]):
                k = f"p{j}_u{u_i}"
                if k not in keys: continue
                zb = pool_to(f[k], GRID)
                if len(zb) < FUT + 1: continue
                zb = ((zb[1:FUT + 1].astype(np.float32) - Z_MU) / Z_SD).astype(np.float16)
                Zs.append(np.concatenate([ctx, zb]))
                As.append(np.concatenate([ah, np.repeat(u["cmd"][None], FUT + 1, 0)]))
                d = u["d"]   # label of future frame kk = danger from that moment to the end of the branch
                Ys.append(np.array([float(d[FRAME_STRIDE * kk:].any()) if len(d) > FRAME_STRIDE * kk else np.nan
                                    for kk in range(1, FUT + 1)], np.float32))
                meta.append((i, j, u_i))
    if not Zs: return None
    return dict(Z=np.stack(Zs), A=np.stack(As).astype(np.float32), Y=np.stack(Ys), meta=meta)

BR = {s: build_branch_seqs(IDS[s]) for s in ["train", "val"]}
for s, b in BR.items(): print(s, "branch sequences:", 0 if b is None else len(b["Z"]))

def get_br_batch(b, n, L):
    si = np.random.randint(0, len(b["Z"]), n); off = np.random.randint(0, b["Z"].shape[1] - L + 1, n)
    Z = np.stack([b["Z"][a, o:o + L] for a, o in zip(si, off)]).astype(np.float32)
    A = np.stack([na(b["A"][a, o:o + L]) for a, o in zip(si, off)]).astype(np.float32)
    return torch.from_numpy(Z).to(DEV), torch.from_numpy(A).to(DEV)

@torch.no_grad()
def val_error_br(m, n=1024):
    # 14-step (2.8 s) imagination error on validation branches, with the branch command
    b = BR["val"]
    if b is None: return float("nan")
    m.eval(); errs = []
    for s in range(0, min(n, len(b["Z"])), 128):
        Z = torch.from_numpy(b["Z"][s:s + 128].astype(np.float32)).to(DEV)
        A = torch.from_numpy(na(b["A"][s:s + 128]).astype(np.float32)).to(DEV)
        errs.append(F.mse_loss(rollout(m, Z[:, :CTX], A), Z[:, CTX:]).item())
    return float(np.mean(errs))

def train_predictor_br(mode="real", epochs=15, patience=3, lr=3e-4, bs=None, rollout_w=1.0):
    bs = bs or TRAIN_BS or max(16, 512 // P); nb = int(bs * BR_FRAC); nm = bs - nb
    torch.manual_seed(SEED); m = Predictor(D, P).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.05)
    best, state, bad = 1e9, None, 0
    for ep in range(epochs):
        m.train(); perm = np.random.permutation(len(TRAIN_STARTS)); t0 = time.time()
        for s in range(len(TRAIN_STARTS) // bs):
            Zm, Am = get_batch([TRAIN_STARTS[j] for j in perm[s * nm:(s + 1) * nm]], L_TRAIN)
            Zb, Ab = get_br_batch(BR["train"], nb, L_TRAIN)
            Z, A = torch.cat([Zm, Zb]), torch.cat([Am, Ab])
            if mode == "shuffled": A = A[torch.randperm(len(A), device=DEV)]
            if mode == "none": A = torch.zeros_like(A)
            loss = F.mse_loss(m(Z, A)[:, :-1], Z[:, 1:]) + rollout_w * F.mse_loss(rollout(m, Z[:, :CTX], A, grad=True), Z[:, CTX:])
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        e_main, e_br = val_error(m, mode), val_error_br(m)
        err = (e_main + e_br) / 2; flag = ""
        if err < best: best, state, bad, flag = err, {k: v.clone() for k, v in m.state_dict().items()}, 0, " *"
        else: bad += 1
        print(f"[{mode}+branches] epoch {ep:02d}  val main {e_main:.4f} | val branches {e_br:.4f}{flag}  ({time.time() - t0:.0f}s)")
        if bad >= patience: break
    m.load_state_dict(state); m.eval(); return m

print("old WM (main only) | val branches error:", round(val_error_br(WM["real"]), 4))
WM_OLD = dict(WM)
WM["real"] = train_predictor_br("real")
WM["shuffled"] = train_predictor_br("shuffled")
print("new WM (main + branches) | val branches error:", round(val_error_br(WM["real"]), 4))

## 11. Level 2: prediction error vs horizon, and the action controls

In [ ]:
TEST_STARTS = window_starts(IDS["test"], CTX + H, step=5)
@torch.no_grad()
def horizon_errors(m, mode):
    errs = []
    for s in range(0, len(TEST_STARTS), 128):
        Z, A = get_batch(TEST_STARTS[s:s + 128], CTX + H, zero_actions=(mode == "none"))
        errs.append(((rollout(m, Z[:, :CTX], A) - Z[:, CTX:]) ** 2).mean((2, 3)).cpu().numpy())
    return np.concatenate(errs).mean(0)
copy = []
for s in range(0, len(TEST_STARTS), 128):
    Z, _ = get_batch(TEST_STARTS[s:s + 128], CTX + H)
    copy.append(((Z[:, CTX - 1:CTX] - Z[:, CTX:]) ** 2).mean((2, 3)).cpu().numpy())
ERR = {"copy-last": np.concatenate(copy).mean(0)}
for mode in WM: ERR[mode] = horizon_errors(WM[mode], mode)
print(f"{len(TEST_STARTS)} test windows\nstep  time " + "".join(f"{k:>11s}" for k in ERR))
for k in range(H): print(f"+{k + 1:2d}  {(k + 1) * DT:4.1f}s" + "".join(f"{ERR[n][k]:11.4f}" for n in ERR))
g = 1 - ERR["real"][-1] / ERR["shuffled"][-1]
print(f"\nactions reduce the 3 s error by {g:.1%} (real vs shuffled)")
RESULTS.append(dict(model="L2 action gain at 3 s", value=float(g)))

## 12. Danger head for planning (real + imagined futures + imagined branch futures)
The planner reads danger on *imagined* latents, which are smoother than real ones. This head is trained on real latents, futures imagined by the world model, and imagined branch futures labelled with the real branch outcome.

In [ ]:
# ===== 12. Risk probe for planning: real latents + imagined futures + imagined BRANCH futures =====
@torch.no_grad()
def imagined(starts, m):
    Zp, Yp = [], []
    for s in range(0, len(starts), 128):
        st = starts[s:s + 128]; Z, A = get_batch(st, CTX + H)
        Zp.append(rollout(m, Z[:, :CTX], A).half().cpu().numpy())
        Yp.append(np.stack([EPS[i]["y5"][t + CTX:t + CTX + H] for i, t in st]))
    return np.concatenate(Zp), np.concatenate(Yp)

@torch.no_grad()
def imagined_br(b, m):
    Zp = []
    for s in range(0, len(b["Z"]), 128):
        Z = torch.from_numpy(b["Z"][s:s + 128, :CTX].astype(np.float32)).to(DEV)
        A = torch.from_numpy(na(b["A"][s:s + 128]).astype(np.float32)).to(DEV)
        Zp.append(rollout(m, Z, A).half().cpu().numpy())
    return np.concatenate(Zp)

tr_st = window_starts(IDS["train"], CTX + H, step=6)
Zi, Yi = imagined(tr_st, WM["real"])
Zbi = imagined_br(BR["train"], WM["real"])
Xr, Yr, _, _ = frames_xy(IDS["train"])
sel = np.random.default_rng(0).choice(len(Xr), size=min(len(Xr), Zi.shape[0] * 4), replace=False)
Xmix = np.concatenate([Xr[sel], Zi.reshape(-1, P, D), Zbi.reshape(-1, P, D)])
Ymix = np.concatenate([Yr[sel], Yi.reshape(-1), BR["train"]["Y"].reshape(-1)])
ok = ~np.isnan(Ymix); Xmix, Ymix = Xmix[ok], Ymix[ok]
Zv, Yv = imagined(VAL_STARTS, WM["real"]); Zbv = imagined_br(BR["val"], WM["real"])
Xv = np.concatenate([Zv.reshape(-1, P, D), Zbv.reshape(-1, P, D)]); Yv_ = np.concatenate([Yv.reshape(-1), BR["val"]["Y"].reshape(-1)])
okv = ~np.isnan(Yv_)
PROBE_PLAN = train_probe(Xmix, Ymix, Xv[okv], Yv_[okv])
del Xr, Xmix
yb = BR["val"]["Y"].reshape(-1); okb = ~np.isnan(yb)
if len(set(yb[okb])) > 1:
    RESULTS.append(report("risk | imagined val branches (plan probe)", yb[okb], probe_predict(PROBE_PLAN, Zbv.reshape(-1, P, D)[okb])))

Zt, Yt = imagined(TEST_STARTS, WM["real"])
for k in [int(1 / DT) - 1, int(2 / DT) - 1, H - 1]:
    y = Yt[:, k]; ok = ~np.isnan(y)
    real_f = np.stack([EPS[i]["zn"][t + CTX + k] for i, t in TEST_STARTS])
    now    = np.stack([EPS[i]["zn"][t + CTX - 1] for i, t in TEST_STARTS])
    tag = f"{(k + 1) * DT:.0f}s"
    RESULTS.append(report(f"risk @{tag} | imagined future (plan probe)", y[ok], probe_predict(PROBE_PLAN, Zt[ok, k])))
    RESULTS.append(report(f"risk @{tag} | current frame (real probe)", y[ok], probe_predict(PROBE_REAL, now[ok])))
    RESULTS.append(report(f"risk @{tag} | real future (real probe, upper bound)", y[ok], probe_predict(PROBE_REAL, real_f[ok])))

## 13. Action sensitivity by distance (real vs shuffled world model)

In [ ]:
@torch.no_grad()
def risk_of_sequence(m, z_ctx, a_hist, cmds, probe):
    # z_ctx (1, CTX, P, D) tensor; a_hist (CTX, 3) raw commands of the context frames (the last one is replaced by
    # the candidate: position t uses a_t to predict z_{t+1}); cmds (N, H, 3) raw -> (N,) max imagined risk over 3 s
    N = len(cmds)
    A = np.concatenate([np.repeat(a_hist[None, :-1], N, 0), cmds, cmds[:, -1:]], 1)
    ro = rollout(m, z_ctx.expand(N, -1, -1, -1), torch.from_numpy(na(A)).to(DEV))
    r = torch.sigmoid(probe(ro.reshape(N * H, P, D))).reshape(N, H)
    return r.max(1).values.cpu().numpy()

names = list(PRIMITIVES); CMDS = np.stack([np.repeat(np.array(PRIMITIVES[k], np.float32)[None], H, 0) for k in names])
bins = [(-1e9, 1, "< 1 m"), (1, 3, "1-3 m"), (3, 6, "3-6 m"), (6, 1e9, "> 6 m")]
sub = TEST_STARTS[::2][:1500]
for mode in ["real", "shuffled"]:
    R = []; cl = []
    for i, t in sub:
        e = EPS[i]
        zc = torch.from_numpy(e["zn"][t:t + CTX].astype(np.float32))[None].to(DEV)
        R.append(risk_of_sequence(WM[mode], zc, e["act5"][t:t + CTX], CMDS, PROBE_PLAN)); cl.append(e["clear5"][t + CTX - 1])
    R = np.array(R); cl = np.array(cl); spread = R.max(1) - R.min(1); fs = R[:, names.index("forward")] - R[:, names.index("wait")]
    print(f"\n== world model trained with {mode} commands ==")
    print("mean risk per command:", {k: round(float(v), 3) for k, v in zip(names, R.mean(0))})
    for lo, hi, nm in bins:
        mk = (cl >= lo) & (cl < hi)
        if mk.sum() > 20:
            print(f"clearance {nm:>6s} (n={mk.sum():4d}) | spread {spread[mk].mean():.3f} | >0.2 {(spread[mk] > 0.2).mean():6.1%} | forward - wait {fs[mk].mean():+.3f}")

## 14. Counterfactual prediction (branches)
For a test branch point, the world model imagines the future of each of the 7 commands. Does the imagined future of command *k* look most like the **real** branch of command *k*? Chance = 1/7.

In [ ]:
BR_FEAT = {}
if ENCODE_BRANCHES:
    for i in BRANCHES:
        p_ = os.path.join(FEAT_DIR, EPS[i]["name"] + "_branches.npz")
        if os.path.exists(p_): BR_FEAT[i] = np.load(p_)

def branch_latents(i, j, u_i):
    k = f"p{j}_u{u_i}"
    if i not in BR_FEAT or k not in BR_FEAT[i].files: return None
    z = pool_to(BR_FEAT[i][k], GRID).astype(np.float32)
    return (z - Z_MU) / Z_SD

if BR_FEAT:
    # alignment check: branch frame 0 should look like the main frame at the branch point
    i = next(iter(BR_FEAT)); p0 = BRANCHES[i][0]; e = EPS[i]; t5 = int(np.searchsorted(e["idx"], p0["t0"], "right") - 1)
    zb = branch_latents(i, 0, 0)
    if zb is not None:
        print("alignment: |branch0 - main(t0)| =", round(float(((zb[0] - e["zn"][t5]) ** 2).mean()), 4),
              "| |main(t0+1) - main(t0)| =", round(float(((e["zn"][t5 + 1] - e["zn"][t5]) ** 2).mean()), 4))
    hits, tot = 0, 0
    for i, pts in BRANCHES.items():
        if SPLIT[i] != "test" or i not in BR_FEAT: continue
        e = EPS[i]
        for j, p in enumerate(pts):
            t5 = int(np.searchsorted(e["idx"], p["t0"], "right") - 1)
            if t5 < CTX - 1: continue
            units = [(u_i, u) for u_i, u in enumerate(p["units"]) if u["cmd"] is not None and branch_latents(i, j, u_i) is not None]
            if len(units) < 3: continue
            K = min(H, min(len(branch_latents(i, j, u_i)) for u_i, _ in units) - 1)
            if K < 3: continue
            zc = torch.from_numpy(e["zn"][t5 - CTX + 1:t5 + 1].astype(np.float32))[None].to(DEV)
            cmds = np.stack([np.repeat(u["cmd"][None], K, 0) for _, u in units])
            A = np.concatenate([np.repeat(e["act5"][t5 - CTX + 1:t5][None], len(units), 0), cmds, cmds[:, -1:]], 1)
            with torch.no_grad():
                ro = rollout(WM["real"], zc.expand(len(units), -1, -1, -1), torch.from_numpy(na(A)).to(DEV)).cpu().numpy()
            real = np.stack([branch_latents(i, j, u_i)[1:K + 1] for u_i, _ in units])
            dist = ((ro[:, None] - real[None]) ** 2).mean((2, 3, 4))          # imagined k vs real j
            hits += int((dist.argmin(1) == np.arange(len(units))).sum()); tot += len(units)
    if tot:
        print(f"counterfactual matching accuracy: {hits / tot:.1%} on {tot} (point, command) pairs (chance ~{1 / 7:.0%})")
        RESULTS.append(dict(model="counterfactual matching accuracy", value=hits / tot, n=tot))

## 15. Planning: world model + MPC, evaluated on the test branch points
At each test branch point the planner sees the 8 frames before the point, the robot pose and the goal, and picks a command. We then look at what **really happened** when that command was executed (the branch).

- **WM + MPC (primitives)**: score the 7 branch commands held for 3 s: cost = max imagined risk − λ·progress to the goal.
- **WM + MPPI**: continuous command sequences (256 samples, 3 iterations) inside the Go2 limits; its first command is matched to the nearest of the 7 branch commands to read the real outcome.
- **Baselines**: random command, always wait, always forward, reactive rule (wait if a machine is closer than 2 m, else forward, using the true distance), oracle.
- **Control**: the same MPC with the world model trained on shuffled commands.

Main metric: share of **decisive points** (some commands safe, some not) where the chosen command was safe.

In [ ]:
LAMBDA_PROG = 0.3        # weight of progress to the goal (risk is in [0, 1]); 0 = pure safety
A_LO, A_HI = np.array(A_LOW, np.float32), np.array(A_HIGH, np.float32)

def progress(pose, goal, cmds):
    # unicycle integration of command sequences (N, K, 3) from pose (x, y, yaw) -> distance-to-goal decrease
    if goal is None: return np.zeros(len(cmds))
    x, y, th = [np.full(len(cmds), v, np.float64) for v in pose]
    for k in range(cmds.shape[1]):
        vx, vy, wz = cmds[:, k, 0], cmds[:, k, 1], cmds[:, k, 2]
        x += (vx * np.cos(th) - vy * np.sin(th)) * DT; y += (vx * np.sin(th) + vy * np.cos(th)) * DT; th += wz * DT
    d0 = np.hypot(pose[0] - goal[0], pose[1] - goal[1])
    return d0 - np.hypot(x - goal[0], y - goal[1])

def plan_primitives(m, zc, a_hist, pose, goal):
    risk = risk_of_sequence(m, zc, a_hist, CMDS, PROBE_PLAN)
    cost = risk - LAMBDA_PROG * progress(pose, goal, CMDS)
    return int(cost.argmin()), risk

def plan_mppi(m, zc, a_hist, pose, goal, N=256, iters=3, sigma=(0.3, 0.1, 0.3), temp=0.05):
    mean = np.zeros((H, 3), np.float32); sig = np.array(sigma, np.float32)
    for it in range(iters):
        seq = np.clip(mean[None] + np.random.randn(N, H, 3).astype(np.float32) * sig, A_LO, A_HI)
        seq[0] = mean                                                    # keep the current mean
        cost = risk_of_sequence(m, zc, a_hist, seq, PROBE_PLAN) - LAMBDA_PROG * progress(pose, goal, seq)
        w = np.exp(-(cost - cost.min()) / temp); w /= w.sum()
        mean = (w[:, None, None] * seq).sum(0)
    return mean[0]

def nearest_primitive(a, cmds):
    s = np.array([0.5, 0.25, 0.6])                                      # rough scale of each command axis
    return int((((cmds - a[None]) / s) ** 2).sum(1).argmin())

rows = []
test_pts = [(i, j, p) for i, pts in BRANCHES.items() if SPLIT[i] == "test" for j, p in enumerate(pts)]
t_start = time.time()
for n_, (i, j, p) in enumerate(test_pts):
    e = EPS[i]; t5 = int(np.searchsorted(e["idx"], p["t0"], "right") - 1)
    if t5 < CTX - 1: continue
    units = [u for u in p["units"] if u["cmd"] is not None]
    if len(units) < 2: continue
    cmds_pt = np.stack([u["cmd"] for u in units]); unsafe = np.array([u["unsafe"] for u in units])
    zc = torch.from_numpy(e["zn"][t5 - CTX + 1:t5 + 1].astype(np.float32))[None].to(DEV)
    a_hist = e["act5"][t5 - CTX + 1:t5 + 1]; pose = (*e["pos"][p["t0"]], e["yaw"][p["t0"]]); goal = e["goal"]
    seq_pt = np.repeat(cmds_pt[:, None], H, 1)
    risk_r = risk_of_sequence(WM["real"], zc, a_hist, seq_pt, PROBE_PLAN)
    risk_s = risk_of_sequence(WM["shuffled"], zc, a_hist, seq_pt, PROBE_PLAN)
    prog = progress(pose, goal, seq_pt)
    pick = {
        "WM + MPC (primitives)": int((risk_r - LAMBDA_PROG * prog).argmin()),
        "control: shuffled WM + MPC": int((risk_s - LAMBDA_PROG * prog).argmin()),
        "WM + MPPI": nearest_primitive(plan_mppi(WM["real"], zc, a_hist, pose, goal), cmds_pt),
        "always wait": nearest_primitive(np.zeros(3), cmds_pt),
        "always forward": nearest_primitive(np.array(PRIMITIVES["forward"]), cmds_pt),
        "reactive (true distance)": nearest_primitive(np.zeros(3) if e["clear"][p["t0"]] < 2.0 else np.array(PRIMITIVES["forward"]), cmds_pt),
    }
    row = dict(ep=i, t0=p["t0"], behav=e["behav"], scen=e["scen"], n_unsafe=int(unsafe.sum()), n_cmd=len(units),
               decisive=bool(0 < unsafe.sum() < len(units)), random=float(1 - unsafe.mean()), oracle=float((~unsafe).any()))
    for k, v in pick.items(): row[k] = float(not unsafe[v])
    row["_risk"] = risk_r; row["_unsafe"] = unsafe
    rows.append(row)
    if n_ % 100 == 0: print(f"{n_}/{len(test_pts)} points  {time.time() - t_start:.0f}s")

import pandas as pd
PLAN = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in rows])
methods = ["random", "always wait", "always forward", "reactive (true distance)", "control: shuffled WM + MPC",
           "WM + MPC (primitives)", "WM + MPPI", "oracle"]
if len(PLAN) == 0:
    raise RuntimeError("no test branch points: fix list_branches() in section 5 (send me the CHECK output)")
dec = PLAN[PLAN.decisive]
print(f"\n{len(PLAN)} test branch points, {len(dec)} decisive")
print("share of decisive points where the chosen command was SAFE:")
for k in methods: print(f"  {k:30s} {dec[k].mean():.3f}")
print("\nper behaviour (WM + MPC primitives vs always wait, decisive points):")
print(dec.groupby("behav")[["WM + MPC (primitives)", "always wait", "random"]].mean().round(3).to_string())
yr = np.concatenate([r["_unsafe"] for r in rows]); pr = np.concatenate([r["_risk"] for r in rows])
if len(set(yr)) > 1:
    RESULTS.append(report("planning | imagined risk vs real branch outcome", yr.astype(float), pr))
for k in methods: RESULTS.append(dict(model=f"planning | {k}", value=float(dec[k].mean()), n=len(dec)))

## 16. Save results and the planner bundle (for the closed loop in Isaac)

In [ ]:
pd.DataFrame(RESULTS).to_csv(os.path.join(RESULTS_DIR, f"results_{ENCODER}_g{GRID}.csv"), index=False)
PLAN.to_csv(os.path.join(RESULTS_DIR, f"planning_branches_{ENCODER}_g{GRID}.csv"), index=False)
torch.save(dict(predictor=WM["real"].state_dict(), probe_plan=PROBE_PLAN.state_dict(), probe_real=PROBE_REAL.state_dict(),
                Z_MU=Z_MU, Z_SD=Z_SD, A_MU=A_MU, A_SD=A_SD,
                cfg=dict(ENCODER=ENCODER, IMG=IMG, GRID=GRID, GRID_STORE=GRID_STORE, CTX=CTX, H=H, FRAME_STRIDE=FRAME_STRIDE,
                         P=int(P), D=int(D), A_LOW=A_LOW, A_HIGH=A_HIGH, LAMBDA_PROG=LAMBDA_PROG)),
           os.path.join(RESULTS_DIR, f"planner_bundle_{ENCODER}_g{GRID}.pt"))
display(pd.DataFrame(RESULTS)) if IN_COLAB else print(pd.DataFrame(RESULTS).to_string())
print("saved to", RESULTS_DIR)